# 01 - Exploración de Datos
## Clasificación de Marcha Patológica mediante Redes Recurrentes y Self-Attention

**Autor:** Weimar Andres Arenas Gonzalez  
**Curso:** Fundamentos de Deep Learning  
**Dataset:** GIST Pathological Gait Dataset (Jun et al., 2020)  

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WeimarArenas/ProyectoDL20261/blob/main/01%20-%20Exploracion_de_datos.ipynb)

---
**Objetivo:** Explorar y visualizar el dataset de marcha patológica para entender la estructura,
distribución de clases, características de las secuencias temporales y la anatomía de articulaciones.

## 1. Instalación de dependencias y configuración del entorno

In [ ]:
# Verificar si estamos en Google Colab
import sys
IN_COLAB = 'google.colab' in sys.modules
print(f'Ejecutando en Google Colab: {IN_COLAB}')

# Instalar dependencias si es necesario
if IN_COLAB:
    !pip install -q matplotlib seaborn pandas numpy scikit-learn

## 2. Descarga y configuración del dataset

El dataset GIST Pathological Gait se encuentra en GitHub: https://github.com/kooksung/pathological_gait_datasets

**Opción A (recomendada en Colab):** Clonar directamente desde GitHub  
**Opción B:** Subir manualmente a Google Drive y montar

In [ ]:
# Ruta al dataset — directorio Pathological_Gaits/
if IN_COLAB and not os.path.exists('pathological_gait_datasets'):
    os.system('git clone --quiet https://github.com/kooksung/pathological_gait_datasets.git')

DATA_DIR = 'pathological_gait_datasets/Pathological_Gaits'
assert os.path.isdir(DATA_DIR), f'No se encontro el dataset: {DATA_DIR}'
n_dirs = len([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
print(f'Dataset encontrado: {DATA_DIR}')
print(f'Nro. de sesiones: {n_dirs}')


## 3. Importación de librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import re
import os
from collections import Counter
from mpl_toolkits.mplot3d import Axes3D

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('Librerías importadas correctamente')

## 4. Funciones de carga de datos

El dataset contiene archivos CSV con el formato:  
`time, joint0_x, joint0_y, joint0_z, ..., joint24_x, joint24_y, joint24_z`  
Cada secuencia tiene entre 80-90 frames; se usan los frames 10-60 (50 frames útiles).

In [ ]:
# ============================================================
# FUNCIONES DE CARGA — Formato real del dataset GIST
# CSV: tab-separado, sin header, 101 columnas
# Col 0: timestamp | cols (1+j*4)+1,+2,+3 = x,y,z para joint j
# ============================================================

import re as _re

FEATURE_COLS = []
for _j in range(25):
    _base = 1 + _j * 4
    FEATURE_COLS.extend([_base + 1, _base + 2, _base + 3])

_GAIT_PAT = _re.compile(
    r'human(\d+)_(normal|antalgic|lurch|steppage|stiff_legged|trendelenburg)\d+'
)
CLASS_MAP = {
    'normal': 0, 'antalgic': 1, 'stiff_legged': 2,
    'lurch': 3, 'steppage': 4, 'trendelenburg': 5,
}
CLASS_NAMES = ['Normal', 'Antalgic', 'Stiff-legged', 'Lurching', 'Steppage', 'Trendelenburg']


def extract_label_and_subject(filepath):
    dirname = os.path.basename(os.path.dirname(filepath))
    m = _GAIT_PAT.match(dirname)
    if not m:
        return None, None
    return CLASS_MAP.get(m.group(2)), int(m.group(1))


def load_csv_sequence(filepath, skip_frames=10, n_frames=50):
    try:
        df = pd.read_csv(filepath, header=None, sep='\t')
        if df.shape[1] != 101 or len(df) < skip_frames + n_frames:
            return None
        seq = df.iloc[skip_frames:skip_frames + n_frames, FEATURE_COLS].values.astype(np.float32)
        return None if np.any(np.isnan(seq)) else seq
    except Exception:
        return None


def load_full_dataset(data_dir, verbose=True):
    """Retorna X (N,50,75), y (N,), S (N,), file_list (N,)."""
    sequences, labels, subjects, file_list = [], [], [], []
    skipped = 0
    for dirname in sorted(os.listdir(data_dir)):
        dirpath = os.path.join(data_dir, dirname)
        if not os.path.isdir(dirpath):
            continue
        for fname in sorted(os.listdir(dirpath)):
            if not fname.endswith('.csv'):
                continue
            fpath = os.path.join(dirpath, fname)
            label, subject = extract_label_and_subject(fpath)
            if label is None:
                skipped += 1
                continue
            seq = load_csv_sequence(fpath)
            if seq is None:
                skipped += 1
                continue
            sequences.append(seq)
            labels.append(label)
            subjects.append(subject)
            file_list.append(fpath)
    if verbose:
        print(f'Secuencias cargadas: {len(sequences)} | Descartadas: {skipped}')
    return (np.array(sequences, dtype=np.float32),
            np.array(labels, dtype=np.int32),
            np.array(subjects, dtype=np.int32),
            file_list)


In [ ]:
# ---- Cargar dataset (definicion de funciones en celda anterior) -------------
X, y, S, file_list = load_full_dataset(DATA_DIR)
print(f'
Forma final: X={X.shape}, y={y.shape}, S={S.shape}')
print(f'Clases: {dict(zip(*__import__("numpy").unique(y, return_counts=True)))}')
print(f'Sujetos: {sorted(__import__("numpy").unique(S).tolist())}')

# Guardar para notebooks siguientes
import numpy as _np
_np.save('X_raw.npy', X)
_np.save('y_raw.npy', y)
_np.save('S_subjects.npy', S)
print('Guardado: X_raw.npy, y_raw.npy, S_subjects.npy')


## 5. Exploración de la estructura del dataset

In [ ]:
# Mostrar primeros archivos para entender la estructura de directorios
print('Primeras 15 rutas de archivos cargados:')
for f in file_list[:15]:
    print(f'  {f}')

In [ ]:
# Examinar un archivo CSV individual para entender el formato
sample_file = file_list[0]
df_sample = pd.read_csv(sample_file)
print(f'Archivo: {sample_file}')
print(f'Forma: {df_sample.shape}')
print(f'Columnas (primeras 10): {list(df_sample.columns[:10])}')
print(f'\nPrimeras 3 filas:')
df_sample.head(3)

## 6. Distribución de clases

In [ ]:
# Conteo de muestras por clase
class_counts = Counter(y)
print('Distribución de clases:')
print('-' * 45)
for cls_id in sorted(class_counts.keys()):
    pct = 100 * class_counts[cls_id] / len(y)
    print(f'  Clase {cls_id} - {CLASS_NAMES[cls_id]:25s}: {class_counts[cls_id]:5d} ({pct:.1f}%)')
print('-' * 45)
print(f'  Total: {len(y)}')
print(f'\n¿Dataset balanceado? {"Sí" if len(set(class_counts.values())) == 1 else "No"}')

In [ ]:
# Visualización de distribución de clases
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras
class_labels = [CLASS_NAMES[i] for i in sorted(class_counts.keys())]
counts = [class_counts[i] for i in sorted(class_counts.keys())]
colors = plt.cm.Set2(np.linspace(0, 1, 6))

bars = axes[0].bar(class_labels, counts, color=colors, edgecolor='black', linewidth=0.8)
axes[0].set_title('Distribución de Clases de Marcha', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Número de Secuencias')
axes[0].set_xlabel('Tipo de Marcha')
axes[0].tick_params(axis='x', rotation=30)
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(count), ha='center', va='bottom', fontsize=10)

# Gráfico de pastel
axes[1].pie(counts, labels=class_labels, colors=colors,
            autopct='%1.1f%%', startangle=90, pctdistance=0.85)
axes[1].set_title('Proporción de Clases', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('distribucion_clases.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Distribución por sujeto y clase (heatmap)
subjects_unique = np.unique(S)
heatmap_data = np.zeros((len(subjects_unique), 6), dtype=int)

for i, subj in enumerate(subjects_unique):
    mask = S == subj
    for cls in range(6):
        heatmap_data[i, cls] = np.sum((y == cls) & mask)

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(heatmap_data, cmap='YlOrRd', aspect='auto')
plt.colorbar(im, ax=ax, label='Número de secuencias')

ax.set_xticks(range(6))
ax.set_xticklabels([CLASS_NAMES[i] for i in range(6)], rotation=25, ha='right')
ax.set_yticks(range(len(subjects_unique)))
ax.set_yticklabels([f'Sujeto {s}' for s in subjects_unique])
ax.set_title('Secuencias por Sujeto y Clase de Marcha', fontsize=13, fontweight='bold')

# Añadir valores en el heatmap
for i in range(len(subjects_unique)):
    for j in range(6):
        ax.text(j, i, str(heatmap_data[i, j]), ha='center', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('heatmap_sujetos_clases.png', dpi=100, bbox_inches='tight')
plt.show()

## 7. Visualización de secuencias temporales

Se visualiza cómo varían las coordenadas de articulaciones clave a lo largo del tiempo para cada tipo de marcha.

In [ ]:
# Seleccionar articulaciones clave para visualización
# SpineBase(0), HipLeft(12), KneeLeft(13), AnkleLeft(14), HipRight(16), KneeRight(17)
KEY_JOINTS = [0, 12, 13, 14, 16, 17]
KEY_JOINT_NAMES = [JOINT_NAMES[j] for j in KEY_JOINTS]

fig, axes = plt.subplots(6, 1, figsize=(14, 18))

for cls_id in range(6):
    # Tomar la primera secuencia de esta clase
    idx = np.where(y == cls_id)[0][0]
    seq = X[idx]  # (50, 75)
    
    ax = axes[cls_id]
    colors_joints = plt.cm.tab10(np.linspace(0, 1, len(KEY_JOINTS)))
    
    for jj, (joint_idx, jname) in enumerate(zip(KEY_JOINTS, KEY_JOINT_NAMES)):
        # Coordenada Y de la articulación (índice de columna: joint_idx*3 + 1)
        y_coord = seq[:, joint_idx * 3 + 1]  # Eje Y (vertical)
        ax.plot(y_coord, label=jname, color=colors_joints[jj], linewidth=1.5)
    
    ax.set_title(f'Clase {cls_id}: {CLASS_NAMES[cls_id]} — Coordenada Y de articulaciones clave',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Frame')
    ax.set_ylabel('Posición Y (m)')
    ax.legend(loc='right', fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Evolución temporal de articulaciones clave por tipo de marcha', 
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('secuencias_temporales.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Comparación directa entre marcha normal y patológicas
# Articulación: KneeLeft (13), coordenada Z (profundidad)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

JOINT_COMPARE = 13  # KneeLeft
N_SAMPLES_OVERLAY = 5  # Superponer 5 secuencias por clase

for cls_id in range(6):
    indices = np.where(y == cls_id)[0][:N_SAMPLES_OVERLAY]
    ax = axes[cls_id]
    
    for idx in indices:
        # Coordenada Z de la rodilla izquierda
        z_coord = X[idx, :, JOINT_COMPARE * 3 + 2]
        ax.plot(z_coord, alpha=0.6, linewidth=1.2)
    
    ax.set_title(f'{CLASS_NAMES[cls_id]}', fontsize=11, fontweight='bold',
                 color='green' if cls_id == 0 else 'red')
    ax.set_xlabel('Frame')
    ax.set_ylabel('Z - KneeLeft (m)')
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Trayectoria de {JOINT_NAMES[JOINT_COMPARE]} (eje Z) — {N_SAMPLES_OVERLAY} secuencias por clase',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('comparacion_clases_knee.png', dpi=100, bbox_inches='tight')
plt.show()

## 8. Análisis estadístico del dataset

In [ ]:
# Estadísticas descriptivas del tensor completo
print('=== ESTADÍSTICAS GLOBALES DEL DATASET ===')
print(f'Forma del tensor: {X.shape}')
print(f'Tipo de datos: {X.dtype}')
print(f'Rango de valores: [{X.min():.4f}, {X.max():.4f}]')
print(f'Media global: {X.mean():.4f}')
print(f'Desviación estándar: {X.std():.4f}')
print(f'NaN en dataset: {np.isnan(X).sum()}')
print(f'Inf en dataset: {np.isinf(X).sum()}')

# Memoria aproximada en RAM
mem_mb = X.nbytes / (1024**2)
print(f'\nMemoria ocupada: {mem_mb:.1f} MB')

In [ ]:
# Distribución de valores por eje de coordenadas
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axis_labels = ['X (lateral)', 'Y (vertical)', 'Z (profundidad)']
colors_axes = ['#2196F3', '#4CAF50', '#FF9800']

for ax_idx in range(3):
    # Extraer todas las coordenadas del eje ax_idx para todos los joints
    coords = X[:, :, ax_idx::3].flatten()
    
    axes[ax_idx].hist(coords, bins=80, color=colors_axes[ax_idx], alpha=0.8, edgecolor='black', linewidth=0.3)
    axes[ax_idx].set_title(f'Distribución Eje {axis_labels[ax_idx]}', fontweight='bold')
    axes[ax_idx].set_xlabel('Valor de coordenada (m)')
    axes[ax_idx].set_ylabel('Frecuencia')
    axes[ax_idx].axvline(np.mean(coords), color='red', linestyle='--', label=f'Media={np.mean(coords):.3f}')
    axes[ax_idx].legend()

plt.suptitle('Distribución de coordenadas por eje', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('distribucion_coordenadas.png', dpi=100, bbox_inches='tight')
plt.show()

## 9. Visualización del esqueleto 3D

In [ ]:
# Conexiones del esqueleto Kinect v2 para visualización
SKELETON_CONNECTIONS = [
    (0, 1), (1, 20), (20, 2), (2, 3),      # Columna + cuello + cabeza
    (20, 4), (4, 5), (5, 6), (6, 7),        # Brazo izquierdo
    (20, 8), (8, 9), (9, 10), (10, 11),     # Brazo derecho
    (0, 12), (12, 13), (13, 14), (14, 15),  # Pierna izquierda
    (0, 16), (16, 17), (17, 18), (18, 19),  # Pierna derecha
    (7, 21), (7, 22), (11, 23), (11, 24),   # Manos
]

def plot_skeleton_3d(seq, frame_idx=25, ax=None, color='steelblue', title=''):
    """Visualiza el esqueleto 3D en un frame específico."""
    if ax is None:
        fig = plt.figure(figsize=(6, 6))
        ax = fig.add_subplot(111, projection='3d')
    
    frame = seq[frame_idx]  # (75,)
    # Reshape a (25, 3): cada fila es (x, y, z) de una articulación
    joints = frame.reshape(25, 3)
    
    # Plotear articulaciones
    ax.scatter(joints[:, 0], joints[:, 2], joints[:, 1],
               c=color, s=30, depthshade=True, zorder=5)
    
    # Plotear conexiones del esqueleto
    for (j1, j2) in SKELETON_CONNECTIONS:
        if j1 < len(joints) and j2 < len(joints):
            ax.plot([joints[j1, 0], joints[j2, 0]],
                    [joints[j1, 2], joints[j2, 2]],
                    [joints[j1, 1], joints[j2, 1]],
                    color=color, linewidth=1.5, alpha=0.8)
    
    ax.set_xlabel('X')
    ax.set_ylabel('Z')
    ax.set_zlabel('Y')
    ax.set_title(title, fontsize=9)
    ax.view_init(elev=15, azim=-60)


# Visualizar un frame del esqueleto para cada tipo de marcha
fig = plt.figure(figsize=(18, 5))
colors_cls = plt.cm.Set1(np.linspace(0, 1, 6))

for cls_id in range(6):
    idx = np.where(y == cls_id)[0][0]
    ax = fig.add_subplot(1, 6, cls_id + 1, projection='3d')
    plot_skeleton_3d(X[idx], frame_idx=25, ax=ax,
                     color=colors_cls[cls_id],
                     title=f'{CLASS_NAMES[cls_id]}')

plt.suptitle('Pose del esqueleto 3D en frame 25 — Todos los tipos de marcha',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('esqueleto_3d_clases.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Animación de la secuencia: mostrar múltiples frames de una clase
# Frame 0, 10, 20, 30, 40 de marcha Normal vs Antálgica
fig = plt.figure(figsize=(18, 8))

FRAMES_TO_SHOW = [0, 10, 20, 30, 40]
CLASSES_TO_SHOW = [0, 1]  # Normal y Antálgica
colors_show = ['#2196F3', '#F44336']

for row, cls_id in enumerate(CLASSES_TO_SHOW):
    idx = np.where(y == cls_id)[0][0]
    for col, frame_idx in enumerate(FRAMES_TO_SHOW):
        ax = fig.add_subplot(2, 5, row * 5 + col + 1, projection='3d')
        plot_skeleton_3d(X[idx], frame_idx=frame_idx, ax=ax,
                         color=colors_show[row],
                         title=f'{CLASS_NAMES[cls_id]}\nFrame {frame_idx}')

plt.suptitle('Evolución del esqueleto: Normal (azul) vs Antálgica (roja)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('evolucion_esqueleto.png', dpi=100, bbox_inches='tight')
plt.show()

## 10. Análisis de variabilidad por articulación

In [ ]:
# Calcular la varianza promedio de cada articulación por clase
# La varianza refleja el movimiento de cada articulación en la secuencia
joint_variance = np.zeros((6, 25))  # (n_classes, n_joints)

for cls_id in range(6):
    mask = y == cls_id
    seqs = X[mask]  # (N_cls, 50, 75)
    
    for j in range(25):
        # Varianza de las 3 coordenadas de la articulación j
        joint_data = seqs[:, :, j*3:(j+1)*3]  # (N_cls, 50, 3)
        joint_variance[cls_id, j] = joint_data.var(axis=(1, 2)).mean()

# Heatmap de varianza
fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(joint_variance, cmap='hot', aspect='auto')
plt.colorbar(im, ax=ax, label='Varianza promedio (m²)')

ax.set_xticks(range(25))
ax.set_xticklabels(JOINT_NAMES, rotation=75, ha='right', fontsize=8)
ax.set_yticks(range(6))
ax.set_yticklabels([CLASS_NAMES[i] for i in range(6)])
ax.set_title('Varianza de articulaciones por tipo de marcha\n'
             '(Mayor varianza = mayor movimiento en esa articulación)',
             fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('varianza_articulaciones.png', dpi=100, bbox_inches='tight')
plt.show()

print('\nArticulaciones con mayor varianza global (más informativas para clasificación):')
mean_var = joint_variance.mean(axis=0)
top_joints = np.argsort(mean_var)[::-1][:10]
for rank, j in enumerate(top_joints, 1):
    print(f'  {rank:2d}. {JOINT_NAMES[j]:20s} (varianza={mean_var[j]:.6f})')

In [ ]:
# Resumen de los grupos de articulaciones relevantes
print('=== GRUPOS DE ARTICULACIONES ===')
print('\nGrupo PIERNAS (usado en el mejor modelo del paper con 93.67%):' )
for j in LEG_JOINT_INDICES:
    print(f'  Joint {j:2d}: {JOINT_NAMES[j]}')

print('\nGrupo BRAZOS (genera ruido según el paper):')
for j in ARM_JOINT_INDICES:
    print(f'  Joint {j:2d}: {JOINT_NAMES[j]}')

print(f'\nFeatures con TODOS los joints: {25 * 3}')
print(f'Features con solo PIERNAS:     {len(LEG_JOINT_INDICES) * 3}')

## 11. Resumen y conclusiones de la exploración

In [ ]:
# Guardar el dataset cargado para uso en notebooks posteriores
np.save('X_raw.npy', X)
np.save('y_raw.npy', y)
np.save('S_subjects.npy', S)

print('=== RESUMEN DE LA EXPLORACIÓN ===')
print(f'\n1. Dataset: GIST Pathological Gait Dataset')
print(f'   - Muestras totales: {len(X)}')
print(f'   - Forma de secuencia: (50 frames, 75 features)')
print(f'   - Clases: {len(CLASS_NAMES)} tipos de marcha')
print(f'   - Dataset balanceado: ~{len(X)//6} muestras por clase')
print(f'   - Sujetos: {len(np.unique(S))} personas')
print(f'\n2. Hallazgos principales:')
print(f'   - Las coordenadas están en metros, sin normalizar')
print(f'   - Las articulaciones de piernas muestran mayor varianza')
print(f'   - Diferencias visuales claras entre Normal y marchas patológicas')
print(f'   - El dataset es adecuado para LOSO-CV con 10 sujetos')
print(f'\n3. Siguiente paso: Preprocesado (notebook 02)')
print(f'   - Normalización por secuencia/global')
print(f'   - Selección de grupos de articulaciones')
print(f'   - Generación de splits LOSO-CV')
print(f'\nArchivos guardados: X_raw.npy, y_raw.npy, S_subjects.npy')